Component reference for the completed project. The final integrated implementation, results, rubric evidence and disclosure are in `final_multi_agent_financial_analysis.ipynb`. Historical execution outputs retain their original provenance.


In [1]:
from pathlib import Path
import sys, json
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from src.news_pipeline import NewsPipeline
from src.data_tools import StockResearchTools
class FixtureClassifier:
    def classify(self, text):
        return "neutral", 1.0
raw = [
    {"title": "Apple synthetic financial report", "summary": "Revenue was 100 in the synthetic example.",
     "publisher": "Synthetic fixture", "link": "https://example.com/aapl-fixture?tracking=1"},
    {"title": "Duplicate fixture", "summary": "This duplicate should be removed.",
     "publisher": "Synthetic fixture", "link": "https://example.com/aapl-fixture"},
]
result = NewsPipeline(classifier=FixtureClassifier()).process("AAPL", raw)
print(json.dumps(result["trace"], indent=2))

{
  "ingest": {
    "input_count": 2
  },
  "preprocess": {
    "retained_count": 1,
    "records": [
      {
        "text": "Apple synthetic financial report. Revenue was 100 in the synthetic example.",
        "url": "https://example.com/aapl-fixture"
      }
    ]
  },
  "classify_finbert": [
    {
      "url": "https://example.com/aapl-fixture",
      "label": "neutral",
      "score": 1.0
    }
  ],
  "extract": [
    {
      "url": "https://example.com/aapl-fixture",
      "facts": [
        "Revenue was 100 in the synthetic example."
      ]
    }
  ],
  "summarize": "Processed 1 source-attributed news items (1 neutral). Representative headlines: Apple synthetic financial report."
}


## Source checks and interpretation
The next checks verify that duplicate URLs collapse, extracted sentences appear in the retained input, and the summary retains the source headline. These checks demonstrate provenance, not independent verification of the article's claims. The production chain receives titles and available summaries, not necessarily full article bodies. Missing bodies and unrelated search results limit the conclusions that can be drawn.

In [2]:
assert len(result["items"]) == 1
item = result["items"][0]
assert item["source"]["url"] == "https://example.com/aapl-fixture"
assert all(fact in result["trace"]["preprocess"]["records"][0]["text"] for fact in item["extracted_facts"])
assert item["title"] in result["summary"]
print("PASS: deduplication, extracted-source alignment, headline-summary alignment")
print(result["summary"])

PASS: deduplication, extracted-source alignment, headline-summary alignment
Processed 1 source-attributed news items (1 neutral). Representative headlines: Apple synthetic financial report.


## Missing data and failure behavior
Yahoo news uses a bounded 20-second HTTP request with a 15-minute successful-result cache, then Finnhub and Alpha Vantage when configured. Historical prices use a 15-second request timeout. These limits are per request, not an overall run deadline. Provider errors remain in failure payloads. Classification failures retain source evidence with sentiment `unavailable`.

A `financial_snapshot` is a provider metric snapshot; a fiscal quarter or reporting period must not be inferred. A sentiment score describes the model's label confidence and does not measure stock-return probability.

In [3]:
class FailedTicker:
    def get_news(self, **kwargs):
        raise TimeoutError("synthetic provider timeout")
empty = NewsPipeline(ticker_factory=lambda _: FailedTicker(), classifier=FixtureClassifier(), fallback_news_sources=[]).process("AAPL")
assert empty["items"] == [] and empty["errors"]
print(json.dumps({"summary": empty["summary"], "errors": empty["errors"]}, indent=2))
class FailedClassifier:
    def classify(self, text):
        raise RuntimeError("synthetic model unavailable")
failed = NewsPipeline(classifier=FailedClassifier()).process("AAPL", raw[:1])
assert failed["items"][0]["sentiment"] == "unavailable"
print(json.dumps({"sentiment": failed["items"][0]["sentiment"], "errors": failed["errors"]}, indent=2))

{
  "summary": "No deduplicated news evidence was available for this run.",
  "errors": [
    "Yahoo Finance news failed: synthetic provider timeout"
  ]
}
{
  "sentiment": "unavailable",
  "errors": [
    "FinBERT classification failed for 'Apple synthetic financial report': synthetic model unavailable"
  ]
}


## Live handoff
Run `StockResearchTools().market_snapshot("AAPL").to_dict()` and `.financial_snapshot("AAPL").to_dict()` for live snapshots, and `NewsPipeline().process("AAPL")` for live retrieval and actual FinBERT classification. The latter downloads model weights on first use. Optional provider keys belong in an untracked `.env`.

The final planner on `codex/final-team-integration` accepts `execute(ticker, research_question, lessons=...)` and returns `evidence`, `draft_analysis`, `plan`, `retrieved_lessons`, and `tool_trace`. The offline integration test verifies lesson delivery to the plan and draft function. Ashley's demonstrations show that a lesson was saved after Run 1 and retrieved before Run 2. The recorded runs did not demonstrate improved analysis quality.

## AI assistance disclosure
Codex assisted Tim with retrieval reliability, integration fixes, offline checks, and preparation of this handoff section. The final report documents the implementation, tests, design decisions and consolidated AI disclosure. Teammates retain authorship of their existing modules. No live inference or final team approval is claimed here.